In [10]:
from dotenv import load_dotenv
import os

# LangChain sınıfları
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# .env dosyasını yükle (gerekirse)
load_dotenv()

PROVIDER = "groq"  # "ollama" | "gemini" | "groq"

if PROVIDER == "ollama":
    llm = ChatOllama(
        model="qwen2.5:7b",
        temperature=0.2
    )
elif PROVIDER == "groq":
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2, api_key=os.getenv("GROQ_API_KEY"))
else:
    llm = ChatGoogleGenerativeAI(google_api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0.2)

# Örnek soru
messages = [HumanMessage(content="Merhaba! Bana kısa bir Python ipucu ver.")]

response = llm.invoke(messages)

print("💡 LLM cevabı:\n", response.content)

💡 LLM cevabı:
 Merhaba. Python'da bir listedeki tüm öğelerin birleştirilmesi için `join()` fonksiyonu kullanılabilir. Örneğin:

```python
liste = ["Merhaba", "Dünya"]
birlesik = " ".join(liste)
print(birlesik)  # Çıktı: Merhaba Dünya
```

Bu kod, `liste` adlı listedeki tüm öğeleri birleştirerek `birlesik` adlı değişkene atar ve sonra bu değişkeni yazdırır. Aralarında bir boşluk bırakmak için `" "` karakterini kullanıyoruz.


In [5]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from functools import lru_cache

load_dotenv()

db_path = r"C:\Projects\LLM-Retail\langchain_esra\data\tshirts.db"
engine = create_engine(f"sqlite:///{db_path}", future=True)

PROVIDER = "groq"  # "ollama" | "gemini" | "groq"

if PROVIDER == "ollama":
    llm_sql = ChatOllama(model="qwen2.5:7b", temperature=0)
elif PROVIDER == "groq":
    llm_sql = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=os.getenv("GROQ_API_KEY"))
else:
    llm_sql = ChatGoogleGenerativeAI(google_api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0)

schema_description = (
    "SQLite tablo yapısı:\n"
    "t_shirts(brand TEXT, color TEXT, size TEXT, price INTEGER, stock_quantity INTEGER)\n"
    "discounts(t_shirt_id INTEGER, pct_discount REAL)\n"
    "Sadece geçerli SQL sorgusunu döndürün. Kod bloğu işaretleri kullanmayın."
)


def normalize_sql(query_text: str) -> str:
    lines = [line.strip() for line in query_text.strip().splitlines()]
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].startswith("```"):
        lines = lines[:-1]
    if lines and lines[0].lower().startswith("sql"):
        lines = [line for line in lines if not line.lower().startswith("sql")]
    return "\n".join(lines).strip()


@lru_cache
def generate_sql(query: str) -> str:
    message = HumanMessage(
        content=(
            f"{schema_description}\n\n"
            f"Sorgu: {query}\n"
            "Yalnızca geçerli SQLite SQL sorgusunu döndürün. Kod bloğu işaretleri veya açıklama ekleme."
        )
    )
    result = llm_sql.invoke([message]).content
    return normalize_sql(result)


def run_sql(query: str):
    sql = generate_sql(query)
    print("Generated SQL:\n", sql)
    with engine.connect() as conn:
        result = conn.execute(text(sql)).all()
    return result

c:\Projects\LLM-Retail\langchain_esra\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Projects\LLM-Retail\langchain_esra\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
query1 = "What is the total stock quantity available for each brand, ordered from highest to lowest?"
result1 = run_sql(query1)
print("Query 1 - Sorgu sonucu:", result1)

Generated SQL:
 SELECT brand, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY brand ORDER BY total_stock DESC
Query 1 - Sorgu sonucu: [('Levi', 1128), ('Nike', 1044), ('Adidas', 1029), ('Van Huesen', 1016)]


In [7]:
query2 = "Which brand has the highest average price per shirt, and what is that average price?"
result2 = run_sql(query2)
print("Query 2 - Sorgu sonucu:", result2)

Generated SQL:
 SELECT brand, AVG(price) as avg_price
FROM t_shirts
GROUP BY brand
ORDER BY avg_price DESC
LIMIT 1
Query 2 - Sorgu sonucu: [('Nike', 33.7)]


In [8]:
query3 = "What is the Total stock per size?"
result3 = run_sql(query3)
print("Query 3 - Sorgu sonucu:", result3)

Generated SQL:
 SELECT size, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY size
Query 3 - Sorgu sonucu: [('L', 760), ('M', 806), ('S', 855), ('XL', 936), ('XS', 860)]


In [9]:
query4 = "What is the total inventory value of all small size t-shirts ?"
result4 = run_sql(query4)
print("Query 4 - Sorgu sonucu:", result4)

Generated SQL:
 SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size = 'small'
Query 4 - Sorgu sonucu: [(None,)]


In [11]:
import pandas as pd
query4 = """
SELECT SUM(price * stock_quantity) AS total_value
FROM t_shirts
WHERE size = 'S'
"""
result4 = pd.read_sql_query(query4, engine)
print("Query 4 - Sorgu sonucu:", result4)

Query 4 - Sorgu sonucu:    total_value
0        26441


In [19]:
query5 = "If we have to sell all Adidas t-shirts, what would be the total revenue with discount?"
result5 = run_sql(query5)
print("Query 5 - Sorgu sonucu:", result5)

Generated SQL:
 SELECT SUM(T1.price * (1 - T2.pct_discount) * T1.stock_quantity)
FROM t_shirts T1
LEFT JOIN discounts T2 ON T1.rowid = T2.t_shirt_id
WHERE T1.brand = 'Adidas'
Query 5 - Sorgu sonucu: [(-731952.0,)]


"How much sales will we generate if we sell all Adidas shirts?"

In [12]:
import pandas as pd
query5 = """
SELECT SUM(t.price * (1 - d.pct_discount / 100) * t.stock_quantity) AS total_discounted_revenue
FROM t_shirts t
JOIN discounts d ON t.t_shirt_id = d.t_shirt_id
WHERE t.brand = 'Adidas'
"""
result5 = pd.read_sql_query(query5, engine)
print("Query 5 - Sorgu sonucu:", result5)

Query 5 - Sorgu sonucu:    total_discounted_revenue
0                  21478.59


In [16]:
query6 = "How many white color Levi t-shirts do we have?"
result6 = run_sql(query6)
print("Query 6 - Sorgu sonucu:", result6)

Generated SQL:
 SELECT stock_quantity FROM t_shirts WHERE color = 'white' AND brand = 'Levi'
Query 6 - Sorgu sonucu: []


In [15]:
import pandas as pd
query6 = """
SELECT stock_quantity FROM t_shirts WHERE color = 'White' AND brand = 'Levi'
"""
result6 = pd.read_sql_query(query6, engine)
print("Query 6 - Sorgu sonucu:", result6)

Query 6 - Sorgu sonucu:    stock_quantity
0              82
1              31
2              60
3              69
4              99
